# Zerobus Healthcare Demo - Interactive Notebook

This notebook demonstrates real-time data ingestion into Databricks Delta Lake using the Zerobus SDK.

**What is Zerobus?**  
Stream data directly into Delta tables via gRPC - no Kafka or message bus needed!

**GitHub**: [bigdatavik/zerobus-healthcare-demo](https://github.com/bigdatavik/zerobus-healthcare-demo)

---
## Configuration

Set your **catalog** in the widget above, then run the cell below. Everything else is auto-detected:
- Workspace URL & ID from Spark config
- Credentials from secret scope `zerobus`

In [ ]:
# =============================================================================
# CONFIGURATION - Set catalog in widget, everything else is auto-detected
# =============================================================================

# Widgets for table location
dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("schema", "zerobus_demo", "Schema")

# Get table config from widgets
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
TABLE = "healthcare_events"
TABLE_NAME = f"{CATALOG}.{SCHEMA}.{TABLE}"

# =============================================================================
# AUTO-DETECT: Workspace URL and ID (works with serverless)
# =============================================================================

# Get workspace URL
WORKSPACE_URL = spark.conf.get("spark.databricks.workspaceUrl", "")
if not WORKSPACE_URL.startswith("https://"):
    WORKSPACE_URL = f"https://{WORKSPACE_URL}"

# Get workspace ID from notebook context (works on serverless)
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
workspace_id = ctx.workspaceId().get()

# Build Zerobus endpoint (AWS us-east-1)
REGION = "us-east-1"
SERVER_ENDPOINT = f"{workspace_id}.zerobus.{REGION}.cloud.databricks.com"

# =============================================================================
# GET CREDENTIALS FROM SECRET SCOPE
# =============================================================================

CLIENT_ID = dbutils.secrets.get(scope="zerobus", key="client_id")
CLIENT_SECRET = dbutils.secrets.get(scope="zerobus", key="client_secret")

# =============================================================================
# DISPLAY CONFIGURATION
# =============================================================================
print("=" * 60)
print("CONFIGURATION (Auto-detected)")
print("=" * 60)
print(f"Workspace URL:    {WORKSPACE_URL}")
print(f"Workspace ID:     {workspace_id}")
print(f"Zerobus Endpoint: {SERVER_ENDPOINT}")
print("-" * 60)
print(f"Table:            {TABLE_NAME}")
print(f"Client ID:        {CLIENT_ID[:8]}...")
print(f"Client Secret:    **********")
print("=" * 60)
print("\n✅ All configuration loaded successfully!")

---
## Demo Reset

Zerobus **APPENDS** data. Run this to reset the table before a demo.

In [ ]:
# Check current record count
print(f"Table: {TABLE_NAME}")
print("\nCurrent record count:")
spark.sql(f"SELECT COUNT(*) as count FROM {TABLE_NAME}").show()

# Uncomment to TRUNCATE (delete all data before demo)
# spark.sql(f"TRUNCATE TABLE {TABLE_NAME}")
# print("✅ Table truncated!")

---
## Architecture Comparison

### Traditional Way (Complex)
```
┌──────────┐    ┌─────────┐    ┌─────────────┐    ┌────────────┐
│  Source  │───▶│  Kafka  │───▶│ Spark Job   │───▶│ Delta Lake │
└──────────┘    └─────────┘    └─────────────┘    └────────────┘
                     ↑               ↑
              Need to manage   Need to manage
```

### Zerobus Way (Simple)
```
┌──────────┐    ┌──────────────────┐    ┌────────────┐
│  Source  │───▶│  Zerobus (gRPC)  │───▶│ Delta Lake │
└──────────┘    └──────────────────┘    └────────────┘
                        ↑
                No infrastructure!
```

## The 5 Key Lines

| Line | Code | What Happens |
|------|------|-------------|
| 1 | `sdk = ZerobusSdk(SERVER_ENDPOINT, WORKSPACE_URL)` | Creates client (no network yet) |
| 2 | `stream = sdk.create_stream(CLIENT_ID, CLIENT_SECRET, ...)` | Opens gRPC + authenticates |
| 3 | `ack = stream.ingest_record(record)` | Sends record over gRPC |
| 4 | `ack.wait_for_ack()` | Waits for durability confirmation |
| 5 | `stream.close()` | Closes the connection |

---
## Real-World Architecture

In production, the producer runs **outside Databricks** and connects via the internet:

```
┌─────────────────────────────────────────────────────────────────────────┐
│                   HEALTHCARE DATA CENTER                                │
│                                                                         │
│  ┌─────────┐   ┌─────────┐   ┌─────────┐   ┌─────────┐                 │
│  │  Epic   │   │ Cerner  │   │  Labs   │   │Pharmacy │                 │
│  └────┬────┘   └────┬────┘   └────┬────┘   └────┬────┘                 │
│       └──────────┬──┴─────────────┴──────────────┘                     │
│                  ▼                                                      │
│       ┌─────────────────────────────────┐                              │
│       │     INTEGRATION SERVER          │                              │
│       │  (Python + Zerobus SDK)         │                              │
│       │                                 │                              │
│       │  stream.ingest_record(event)    │                              │
│       └────────────────┬────────────────┘                              │
└────────────────────────┼────────────────────────────────────────────────┘
                         │ gRPC over HTTPS
                         ▼
┌─────────────────────────────────────────────────────────────────────────┐
│                      DATABRICKS CLOUD                                   │
│                                                                         │
│  ┌───────────────────────────────────────────────────────────────────┐ │
│  │                 ZEROBUS INGEST SERVICE                            │ │
│  │  • Receives gRPC streams     • Validates schema                   │ │
│  │  • Batches records           • Sends durability ACKs              │ │
│  └─────────────────────────────────┬─────────────────────────────────┘ │
│                                    ▼                                    │
│  ┌───────────────────────────────────────────────────────────────────┐ │
│  │                    UNITY CATALOG                                  │ │
│  │  <catalog>.<schema>.healthcare_events (Delta Table)               │ │
│  └───────────────────────────────────────────────────────────────────┘ │
└─────────────────────────────────────────────────────────────────────────┘
```

---
# Step 1: Install SDK

In [ ]:
%pip install databricks-zerobus-ingest-sdk>=0.2.0 -q

# Step 2: Imports

In [ ]:
import json
import time
import uuid
import random

from zerobus.sdk.sync import ZerobusSdk
from zerobus.sdk.shared import RecordType, StreamConfigurationOptions, TableProperties

print("Imports successful!")

# Step 3: Configuration Ready

Configuration was loaded in the first cell. All values are auto-detected from:
- **Workspace URL/ID**: Spark config
- **Credentials**: Secret scope `zerobus`
- **Table**: Widget values

In [ ]:
# Configuration summary
print(f"✅ Server:    {SERVER_ENDPOINT}")
print(f"✅ Workspace: {WORKSPACE_URL}")
print(f"✅ Table:     {TABLE_NAME}")
print(f"✅ Client ID: {CLIENT_ID[:8]}...")

# Step 4: Sample Data Generator

In [ ]:
EVENT_TYPES = ["admission", "discharge", "claim", "rx_fill", "lab_result"]
FACILITY_CODES = ["FAC001", "FAC002", "FAC003", "FAC004", "FAC005"]
DIAGNOSIS_CODES = ["E11.9", "I10", "J06.9", "M54.5", "K21.0"]
PROCEDURE_CODES = ["99213", "99214", "99215", "99203", "99204"]

def generate_healthcare_event():
    event_type = random.choice(EVENT_TYPES)
    amount_ranges = {
        "admission": (5000, 50000), "discharge": (0, 500),
        "claim": (100, 10000), "rx_fill": (10, 500), "lab_result": (50, 1000)
    }
    min_amt, max_amt = amount_ranges.get(event_type, (100, 1000))
    
    return {
        "event_id": str(uuid.uuid4()),
        "member_id": f"MBR{random.randint(100000, 999999)}",
        "event_type": event_type,
        "event_timestamp": int(time.time() * 1_000_000),  # Microseconds!
        "facility_code": random.choice(FACILITY_CODES),
        "diagnosis_code": random.choice(DIAGNOSIS_CODES),
        "procedure_code": random.choice(PROCEDURE_CODES),
        "provider_npi": f"{random.randint(1000000000, 9999999999)}",
        "amount": round(random.uniform(min_amt, max_amt), 2),
        "metadata": json.dumps({"source": "demo", "batch": str(uuid.uuid4())[:8]})
    }

# Test it
sample = generate_healthcare_event()
print("Sample event:")
for k, v in sample.items():
    print(f"  {k}: {v}")

# Step 5: Initialize SDK

No network call yet - just storing config.

In [ ]:
sdk = ZerobusSdk(SERVER_ENDPOINT, WORKSPACE_URL)
print("SDK initialized (no connection yet)")

# Step 6: Create Stream

**THIS is the first network call!**
- Opens gRPC connection
- Authenticates with OAuth
- Validates table access

In [ ]:
options = StreamConfigurationOptions(record_type=RecordType.JSON)
table_props = TableProperties(TABLE_NAME)

print("Connecting to Zerobus...")
stream = sdk.create_stream(CLIENT_ID, CLIENT_SECRET, table_props, options)
print("Connected and authenticated!")

# Step 7: Ingest Records

## Where Do These Methods Come From?

```
┌─────────────────────────────────────────────────────────────────┐
│  pip install databricks-zerobus-ingest-sdk                      │
│                                                                 │
│  ┌───────────────────────────────────────────────────────────┐  │
│  │  ZerobusSdk class                                         │  │
│  │    └── .create_stream() → Stream object                   │  │
│  │                                                           │  │
│  │  Stream class                                             │  │
│  │    ├── .ingest_record(dict) → Ack object     ◄── SEND     │  │
│  │    └── .close()                                           │  │
│  │                                                           │  │
│  │  Ack class                                                │  │
│  │    └── .wait_for_ack()                       ◄── CONFIRM  │  │
│  └───────────────────────────────────────────────────────────┘  │
└─────────────────────────────────────────────────────────────────┘
```

After `wait_for_ack()` returns, the record is **durably written** to Delta Lake.

In [ ]:
num_records = 10
successful = 0

print(f"Ingesting {num_records} healthcare events...")
print("-" * 60)

for i in range(num_records):
    record = generate_healthcare_event()
    
    # Send record over gRPC
    ack = stream.ingest_record(record)
    
    # Wait for durability confirmation
    ack.wait_for_ack()
    
    successful += 1
    print(f"  [{i+1}/{num_records}] {record['event_type']:12} | {record['member_id']} | ${record['amount']:,.2f}")

print("-" * 60)
print(f"Done! {successful} records ingested.")

# Step 8: Close Stream

In [ ]:
stream.close()
print("Stream closed.")

# Step 9: Validate Data

In [ ]:
# Total count
display(spark.sql(f"SELECT COUNT(*) as total_records FROM {TABLE_NAME}"))

In [ ]:
# Recent records
display(spark.sql(f"""
    SELECT event_id, member_id, event_type, 
           from_unixtime(event_timestamp / 1000000) as event_time, amount
    FROM {TABLE_NAME}
    ORDER BY event_timestamp DESC
    LIMIT 10
"""))

In [ ]:
# Analytics
display(spark.sql(f"""
    SELECT event_type, COUNT(*) as count, 
           ROUND(AVG(amount), 2) as avg_amount,
           ROUND(SUM(amount), 2) as total_amount
    FROM {TABLE_NAME}
    GROUP BY event_type
    ORDER BY count DESC
"""))

---
# Summary

## What We Did
1. **Initialized SDK** - No network
2. **Created Stream** - gRPC connection + OAuth
3. **Ingested Records** - Send + wait for ACK
4. **Closed Stream** - Cleanup
5. **Validated** - Queried Delta table

## Key Takeaways
- **No Kafka** - Direct writes to Delta
- **Durability** - `wait_for_ack()` = confirmed write
- **Simple** - 5 lines of code
- **Fast** - Sub-second latency

## Learn More
- [Zerobus Docs](https://docs.databricks.com/aws/en/ingestion/zerobus-overview)
- [GitHub](https://github.com/bigdatavik/zerobus-healthcare-demo)